이 노트북은 [Donne Martin](https://github.com/donnemartin)이 작성했습니다. 원본과 라이선스 정보는 [GitHub](https://github.com/donnemartin/system-design-primer)에 있습니다.

*한국어 번역본입니다. 원문: [online_chat.ipynb](online_chat.ipynb)*

# 온라인 채팅 설계하기

## 제약 조건과 가정

* 다음 워크플로에 집중한다고 가정합니다.
    * 텍스트 대화만 지원
    * 사용자
        * 사용자 추가
        * 사용자 삭제
        * 사용자 정보 수정
        * 사용자의 친구 목록에 추가
            * 친구 요청 추가
                * 친구 요청 수락
                * 친구 요청 거절
        * 사용자의 친구 목록에서 삭제
    * 그룹 채팅 생성
        * 그룹 채팅에 친구 초대
        * 그룹 채팅에 메시지 게시
    * 1:1 개인 채팅
        * 개인 채팅에 친구 초대
        * 개인 채팅에 메시지 게시
* 처음에는 확장성을 신경 쓸 필요 없음

## 해답

In [1]:
%%writefile online_chat.py
from abc import ABCMeta


class UserService(object):

    def __init__(self):
        self.users_by_id = {}  # key: user id, value: User

    def add_user(self, user_id, name, pass_hash):  # ...
    def remove_user(self, user_id):  # ...
    def add_friend_request(self, from_user_id, to_user_id):  # ...
    def approve_friend_request(self, from_user_id, to_user_id):  # ...
    def reject_friend_request(self, from_user_id, to_user_id):  # ...


class User(object):

    def __init__(self, user_id, name, pass_hash):
        self.user_id = user_id
        self.name = name
        self.pass_hash = pass_hash
        self.friends_by_id = {}  # key: friend id, value: User
        self.friend_ids_to_private_chats = {}  # key: friend id, value: private chats
        self.group_chats_by_id = {}  # key: chat id, value: GroupChat
        self.received_friend_requests_by_friend_id = {}  # key: friend id, value: AddRequest
        self.sent_friend_requests_by_friend_id = {}  # key: friend id, value: AddRequest

    def message_user(self, friend_id, message):  # ...
    def message_group(self, group_id, message):  # ...
    def send_friend_request(self, friend_id):  # ...
    def receive_friend_request(self, friend_id):  # ...
    def approve_friend_request(self, friend_id):  # ...
    def reject_friend_request(self, friend_id):  # ...


class Chat(metaclass=ABCMeta):

    def __init__(self, chat_id):
        self.chat_id = chat_id
        self.users = []
        self.messages = []


class PrivateChat(Chat):

    def __init__(self, first_user, second_user):
        super(PrivateChat, self).__init__()
        self.users.append(first_user)
        self.users.append(second_user)


class GroupChat(Chat):

    def add_user(self, user):  # ...
    def remove_user(self, user):  # ... 


class Message(object):

    def __init__(self, message_id, message, timestamp):
        self.message_id = message_id
        self.message = message
        self.timestamp = timestamp


class AddRequest(object):

    def __init__(self, from_user_id, to_user_id, request_status, timestamp):
        self.from_user_id = from_user_id
        self.to_user_id = to_user_id
        self.request_status = request_status
        self.timestamp = timestamp


class RequestStatus(Enum):

    UNREAD = 0
    READ = 1
    ACCEPTED = 2
    REJECTED = 3

Overwriting online_chat.py
